In [ ]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from category_encoders import WOEEncoder

import sys

sys.path.append("../src")
from data_loader import load_data
from data_processing import (
    AggregateFeatures,
    DateFeatures,
    build_pipeline,
    calculate_credit_risk_metrics
)

from xverse.transformer import WOE

import warnings
warnings.filterwarnings("ignore")

In [2]:
df = load_data(
    "../data/data.csv"
)
print("Shape:", df.shape)

df.head()

2026-05-31 13:23:18,853 - INFO - Dataset loaded successfully


Shape: (95662, 16)


,TransactionId,BatchId,AccountId,SubscriptionId,CustomerId,CurrencyCode,CountryCode,ProviderId,ProductId,ProductCategory,ChannelId,Amount,Value,TransactionStartTime,PricingStrategy,FraudResult
0,TransactionId_76871,BatchId_36123,AccountId_3957,SubscriptionId_887,CustomerId_4406,UGX,256,ProviderId_6,ProductId_10,airtime,ChannelId_3,1000.0,1000,2018-11-15T02:18:49Z,2,0
1,TransactionId_73770,BatchId_15642,AccountId_4841,SubscriptionId_3829,CustomerId_4406,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-20.0,20,2018-11-15T02:19:08Z,2,0
2,TransactionId_26203,BatchId_53941,AccountId_4229,SubscriptionId_222,CustomerId_4683,UGX,256,ProviderId_6,ProductId_1,airtime,ChannelId_3,500.0,500,2018-11-15T02:44:21Z,2,0
3,TransactionId_380,BatchId_102363,AccountId_648,SubscriptionId_2185,CustomerId_988,UGX,256,ProviderId_1,ProductId_21,utility_bill,ChannelId_3,20000.0,21800,2018-11-15T03:32:55Z,2,0
4,TransactionId_28195,BatchId_38780,AccountId_4841,SubscriptionId_3829,CustomerId_988,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-644.0,644,2018-11-15T03:34:21Z,2,0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 95662 entries, 0 to 95661
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   TransactionId         95662 non-null  str    
 1   BatchId               95662 non-null  str    
 2   AccountId             95662 non-null  str    
 3   SubscriptionId        95662 non-null  str    
 4   CustomerId            95662 non-null  str    
 5   CurrencyCode          95662 non-null  str    
 6   CountryCode           95662 non-null  int64  
 7   ProviderId            95662 non-null  str    
 8   ProductId             95662 non-null  str    
 9   ProductCategory       95662 non-null  str    
 10  ChannelId             95662 non-null  str    
 11  Amount                95662 non-null  float64
 12  Value                 95662 non-null  int64  
 13  TransactionStartTime  95662 non-null  str    
 14  PricingStrategy       95662 non-null  int64  
 15  FraudResult           95662 no

In [4]:
missing = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Percentage": round(
        (df.isnull().sum()/len(df))*100,
        2
    )
})

missing[missing["Missing Values"] > 0]

,Missing Values,Percentage


In [5]:
agg_transformer = AggregateFeatures()

df_agg = agg_transformer.fit_transform(df)

df_agg[
    [
        "CustomerId",
        "Total_Transaction_Amount",
        "Average_Transaction_Amount",
        "Transaction_Count",
        "Std_Transaction_Amount"
    ]
].head()

,CustomerId,Total_Transaction_Amount,Average_Transaction_Amount,Transaction_Count,Std_Transaction_Amount
0,CustomerId_4406,109921.75,923.712185,119,3042.294251
1,CustomerId_4406,109921.75,923.712185,119,3042.294251
2,CustomerId_4683,1000.00,500.000000,2,0.000000
3,CustomerId_988,228727.20,6019.136842,38,17169.241610
4,CustomerId_988,228727.20,6019.136842,38,17169.241610


In [6]:
date_transformer = DateFeatures()

df_date = date_transformer.fit_transform(df_agg)

df_date[
    [
        "TransactionStartTime",
        "Transaction_Hour",
        "Transaction_Day",
        "Transaction_Month",
        "Transaction_Year"
    ]
].head()

,TransactionStartTime,Transaction_Hour,Transaction_Day,Transaction_Month,Transaction_Year
0,2018-11-15 02:18:49+00:00,2,15,11,2018
1,2018-11-15 02:19:08+00:00,2,15,11,2018
2,2018-11-15 02:44:21+00:00,2,15,11,2018
3,2018-11-15 03:32:55+00:00,3,15,11,2018
4,2018-11-15 03:34:21+00:00,3,15,11,2018


In [ ]:

# Identify categorical columns to encode (excluding IDs and already transformed features)
categorical_cols = [
    'CurrencyCode',
    'ProviderId',
    'ProductId',
    'ProductCategory',
    'ChannelId'
]

# Create a copy to avoid modifying the original df_date during transformations
df_processed = df_date.copy()

# One-Hot Encode selected categorical columns
# Initialize OneHotEncoder
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit and transform the categorical columns
encoded_features = encoder.fit_transform(df_processed[categorical_cols])

# Create a DataFrame with the encoded features
encoded_df = pd.DataFrame(
    encoded_features,
    columns=encoder.get_feature_names_out(categorical_cols),
    index=df_processed.index
)

# Concatenate the new encoded features with the original DataFrame and drop the original categorical columns
df_processed = pd.concat([df_processed.drop(columns=categorical_cols), encoded_df], axis=1)

print(f"Shape after One-Hot Encoding: {df_processed.shape}")
display(df_processed.head())

Shape after One-Hot Encoding: (95662, 63)


,TransactionId,BatchId,AccountId,SubscriptionId,CustomerId,CountryCode,Amount,Value,TransactionStartTime,PricingStrategy,...,ProductCategory_movies,ProductCategory_other,ProductCategory_ticket,ProductCategory_transport,ProductCategory_tv,ProductCategory_utility_bill,ChannelId_ChannelId_1,ChannelId_ChannelId_2,ChannelId_ChannelId_3,ChannelId_ChannelId_5
0,TransactionId_76871,BatchId_36123,AccountId_3957,SubscriptionId_887,CustomerId_4406,256,1000.0,1000,2018-11-15 02:18:49+00:00,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,TransactionId_73770,BatchId_15642,AccountId_4841,SubscriptionId_3829,CustomerId_4406,256,-20.0,20,2018-11-15 02:19:08+00:00,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,TransactionId_26203,BatchId_53941,AccountId_4229,SubscriptionId_222,CustomerId_4683,256,500.0,500,2018-11-15 02:44:21+00:00,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,TransactionId_380,BatchId_102363,AccountId_648,SubscriptionId_2185,CustomerId_988,256,20000.0,21800,2018-11-15 03:32:55+00:00,2,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,TransactionId_28195,BatchId_38780,AccountId_4841,SubscriptionId_3829,CustomerId_988,256,-644.0,644,2018-11-15 03:34:21+00:00,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [ ]:

numerical_cols = [
    'CountryCode',
    'Amount',
    'Value',
    'PricingStrategy',
    'Total_Transaction_Amount',
    'Average_Transaction_Amount',
    'Transaction_Count',
    'Std_Transaction_Amount',
    'Transaction_Hour',
    'Transaction_Day',
    'Transaction_Month',
    'Transaction_Year'
]

# Ensure these columns exist in df_processed before scaling
numerical_cols = [col for col in numerical_cols if col in df_processed.columns]

# Initialize StandardScaler
scaler = StandardScaler()

# Apply StandardScaler to the numerical columns
df_processed[numerical_cols] = scaler.fit_transform(df_processed[numerical_cols])

print(f"Shape after Standardization: {df_processed.shape}")
display(df_processed.head())

Shape after Standardization: (95662, 63)


,TransactionId,BatchId,AccountId,SubscriptionId,CustomerId,CountryCode,Amount,Value,TransactionStartTime,PricingStrategy,...,ProductCategory_movies,ProductCategory_other,ProductCategory_ticket,ProductCategory_transport,ProductCategory_tv,ProductCategory_utility_bill,ChannelId_ChannelId_1,ChannelId_ChannelId_2,ChannelId_ChannelId_3,ChannelId_ChannelId_5
0,TransactionId_76871,BatchId_36123,AccountId_3957,SubscriptionId_887,CustomerId_4406,0.0,-0.046371,-0.072291,2018-11-15 02:18:49+00:00,-0.349252,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,TransactionId_73770,BatchId_15642,AccountId_4841,SubscriptionId_3829,CustomerId_4406,0.0,-0.054643,-0.080251,2018-11-15 02:19:08+00:00,-0.349252,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,TransactionId_26203,BatchId_53941,AccountId_4229,SubscriptionId_222,CustomerId_4683,0.0,-0.050426,-0.076352,2018-11-15 02:44:21+00:00,-0.349252,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,TransactionId_380,BatchId_102363,AccountId_648,SubscriptionId_2185,CustomerId_988,0.0,0.107717,0.096648,2018-11-15 03:32:55+00:00,-0.349252,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,TransactionId_28195,BatchId_38780,AccountId_4841,SubscriptionId_3829,CustomerId_988,0.0,-0.059704,-0.075183,2018-11-15 03:34:21+00:00,-0.349252,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [7]:
y = df_date["FraudResult"]

In [16]:
median_amount = (
    df_date["Total_Transaction_Amount"]
    .median()
)

df_date["is_high_risk"] = (
    df_date["Total_Transaction_Amount"]
    < median_amount
).astype(int)

y = df_date["is_high_risk"]

In [17]:
features_for_woe = [
    "Amount",
    "Value"
]

X_woe = df_date[
    features_for_woe
].copy()

In [18]:
print(type(X_woe))
print(X_woe)

<class 'pandas.DataFrame'>
        Amount  Value
0       1000.0   1000
1        -20.0     20
2        500.0    500
3      20000.0  21800
4       -644.0    644
...        ...    ...
95657  -1000.0   1000
95658   1000.0   1000
95659    -20.0     20
95660   3000.0   3000
95661    -60.0     60

[95662 rows x 2 columns]


In [ ]:

X_woe_binned = X_woe.copy()
for col in X_woe_binned.columns:
    X_woe_binned[col] = pd.cut(X_woe_binned[col], bins=5).astype(str)

# Now apply the WoE encoder to the binned (categorical) data
encoder = WOEEncoder()
X_woe_transformed = encoder.fit_transform(X_woe_binned, y)

In [20]:
# 2. Execute the metrics calculation
X_woe_transformed, iv_table = calculate_credit_risk_metrics(X_woe, y, bins=10)

# 3. Display the Information Value Table for your analysis documentation
print("=== INFORMATION VALUE (IV) TABLE ===")
display(iv_table)

# 4. Show the transformed dataset ready for your model training
print("\n=== TRANSFORMATION PREVIEW ===")
display(X_woe_transformed.head())

=== INFORMATION VALUE (IV) TABLE ===


,Variable,Information Value (IV),Predictive Power
0,Amount,0.199307,Medium
1,Value,0.042546,Weak



=== TRANSFORMATION PREVIEW ===


,Amount_woe,Value_woe
0,0.146879,0.070419
1,0.219148,0.344905
2,0.146879,0.035352
3,0.977030,-0.251168
4,-0.101394,0.070419


In [21]:
pipeline = build_pipeline(df_date)

pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('aggregate_features', ...), ('date_features', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the differ

In [23]:
df.columns.tolist()

['TransactionId',
 'BatchId',
 'AccountId',
 'SubscriptionId',
 'CustomerId',
 'CurrencyCode',
 'CountryCode',
 'ProviderId',
 'ProductId',
 'ProductCategory',
 'ChannelId',
 'Amount',
 'Value',
 'TransactionStartTime',
 'PricingStrategy',
 'FraudResult']

In [24]:
df_processed.columns.tolist()

['TransactionId',
 'BatchId',
 'AccountId',
 'SubscriptionId',
 'CustomerId',
 'CountryCode',
 'Amount',
 'Value',
 'TransactionStartTime',
 'PricingStrategy',
 'FraudResult',
 'Total_Transaction_Amount',
 'Average_Transaction_Amount',
 'Transaction_Count',
 'Std_Transaction_Amount',
 'Transaction_Hour',
 'Transaction_Day',
 'Transaction_Month',
 'Transaction_Year',
 'is_high_risk',
 'CurrencyCode_UGX',
 'ProviderId_ProviderId_1',
 'ProviderId_ProviderId_2',
 'ProviderId_ProviderId_3',
 'ProviderId_ProviderId_4',
 'ProviderId_ProviderId_5',
 'ProviderId_ProviderId_6',
 'ProductId_ProductId_1',
 'ProductId_ProductId_10',
 'ProductId_ProductId_11',
 'ProductId_ProductId_12',
 'ProductId_ProductId_13',
 'ProductId_ProductId_14',
 'ProductId_ProductId_15',
 'ProductId_ProductId_16',
 'ProductId_ProductId_19',
 'ProductId_ProductId_2',
 'ProductId_ProductId_20',
 'ProductId_ProductId_21',
 'ProductId_ProductId_22',
 'ProductId_ProductId_23',
 'ProductId_ProductId_24',
 'ProductId_ProductId_

In [26]:
processed_data = pipeline.fit_transform(df_processed)

processed_data.shape

ValueError: A given column is not a column of the dataframe

In [27]:
agg_transformer = AggregateFeatures()
df_agg = agg_transformer.fit_transform(df)

In [28]:
print(df_agg.columns.tolist())

['TransactionId', 'BatchId', 'AccountId', 'SubscriptionId', 'CustomerId', 'CurrencyCode', 'CountryCode', 'ProviderId', 'ProductId', 'ProductCategory', 'ChannelId', 'Amount', 'Value', 'TransactionStartTime', 'PricingStrategy', 'FraudResult', 'Total_Transaction_Amount', 'Average_Transaction_Amount', 'Transaction_Count', 'Std_Transaction_Amount']


In [29]:
df_agg = AggregateFeatures().fit_transform(df)

print("Aggregate columns exist:",
      "Total_Transaction_Amount" in df_agg.columns)

print(df_agg.columns)

Aggregate columns exist: True
Index(['TransactionId', 'BatchId', 'AccountId', 'SubscriptionId', 'CustomerId',
       'CurrencyCode', 'CountryCode', 'ProviderId', 'ProductId',
       'ProductCategory', 'ChannelId', 'Amount', 'Value',
       'TransactionStartTime', 'PricingStrategy', 'FraudResult',
       'Total_Transaction_Amount', 'Average_Transaction_Amount',
       'Transaction_Count', 'Std_Transaction_Amount'],
      dtype='str')


In [30]:
df_featured = AggregateFeatures().fit_transform(df)
df_featured = DateFeatures().fit_transform(df_featured)

pipeline = build_pipeline(df_featured)

In [31]:
df_featured.head(5)

,TransactionId,BatchId,AccountId,SubscriptionId,CustomerId,CurrencyCode,CountryCode,ProviderId,ProductId,ProductCategory,...,PricingStrategy,FraudResult,Total_Transaction_Amount,Average_Transaction_Amount,Transaction_Count,Std_Transaction_Amount,Transaction_Hour,Transaction_Day,Transaction_Month,Transaction_Year
0,TransactionId_76871,BatchId_36123,AccountId_3957,SubscriptionId_887,CustomerId_4406,UGX,256,ProviderId_6,ProductId_10,airtime,...,2,0,109921.75,923.712185,119,3042.294251,2,15,11,2018
1,TransactionId_73770,BatchId_15642,AccountId_4841,SubscriptionId_3829,CustomerId_4406,UGX,256,ProviderId_4,ProductId_6,financial_services,...,2,0,109921.75,923.712185,119,3042.294251,2,15,11,2018
2,TransactionId_26203,BatchId_53941,AccountId_4229,SubscriptionId_222,CustomerId_4683,UGX,256,ProviderId_6,ProductId_1,airtime,...,2,0,1000.00,500.000000,2,0.000000,2,15,11,2018
3,TransactionId_380,BatchId_102363,AccountId_648,SubscriptionId_2185,CustomerId_988,UGX,256,ProviderId_1,ProductId_21,utility_bill,...,2,0,228727.20,6019.136842,38,17169.241610,3,15,11,2018
4,TransactionId_28195,BatchId_38780,AccountId_4841,SubscriptionId_3829,CustomerId_988,UGX,256,ProviderId_4,ProductId_6,financial_services,...,2,0,228727.20,6019.136842,38,17169.241610,3,15,11,2018


In [33]:
processed_df = pd.DataFrame(
    df_featured
)

processed_df.head()

,TransactionId,BatchId,AccountId,SubscriptionId,CustomerId,CurrencyCode,CountryCode,ProviderId,ProductId,ProductCategory,...,PricingStrategy,FraudResult,Total_Transaction_Amount,Average_Transaction_Amount,Transaction_Count,Std_Transaction_Amount,Transaction_Hour,Transaction_Day,Transaction_Month,Transaction_Year
0,TransactionId_76871,BatchId_36123,AccountId_3957,SubscriptionId_887,CustomerId_4406,UGX,256,ProviderId_6,ProductId_10,airtime,...,2,0,109921.75,923.712185,119,3042.294251,2,15,11,2018
1,TransactionId_73770,BatchId_15642,AccountId_4841,SubscriptionId_3829,CustomerId_4406,UGX,256,ProviderId_4,ProductId_6,financial_services,...,2,0,109921.75,923.712185,119,3042.294251,2,15,11,2018
2,TransactionId_26203,BatchId_53941,AccountId_4229,SubscriptionId_222,CustomerId_4683,UGX,256,ProviderId_6,ProductId_1,airtime,...,2,0,1000.00,500.000000,2,0.000000,2,15,11,2018
3,TransactionId_380,BatchId_102363,AccountId_648,SubscriptionId_2185,CustomerId_988,UGX,256,ProviderId_1,ProductId_21,utility_bill,...,2,0,228727.20,6019.136842,38,17169.241610,3,15,11,2018
4,TransactionId_28195,BatchId_38780,AccountId_4841,SubscriptionId_3829,CustomerId_988,UGX,256,ProviderId_4,ProductId_6,financial_services,...,2,0,228727.20,6019.136842,38,17169.241610,3,15,11,2018


In [34]:
pd.DataFrame(
    processed_df.isnull().sum(),
    columns=["Missing"]
).T

,TransactionId,BatchId,AccountId,SubscriptionId,CustomerId,CurrencyCode,CountryCode,ProviderId,ProductId,ProductCategory,...,PricingStrategy,FraudResult,Total_Transaction_Amount,Average_Transaction_Amount,Transaction_Count,Std_Transaction_Amount,Transaction_Hour,Transaction_Day,Transaction_Month,Transaction_Year
Missing,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,712,0,0,0,0


In [35]:
processed_df.to_csv(
    "../data/processed/processed_credit_data.csv",
    index=False
)

print(
    "Processed dataset saved successfully."
)

Processed dataset saved successfully.
